## 1. Setup & Configuration

In [31]:
import os
import json
import uuid
import random
import gc
import numpy as np
# Non-GUI backend to avoid laptop hang / high memory
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Callable
from dataclasses import dataclass
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")
print("Matplotlib backend:", matplotlib.get_backend(), "(Agg = no GUI, less memory)")

All libraries imported successfully!
Matplotlib backend: Agg (Agg = no GUI, less memory)


In [32]:
@dataclass
class GenerationConfig:
    """Configuration for data generation"""
    output_dir: str = "synthetic_dataset"
    num_images: int = 3000
    seed: int = 42
    
    # Image settings (smaller = less memory, no hang on laptop)
    image_width_range: Tuple[int, int] = (600, 1000)
    image_height_range: Tuple[int, int] = (450, 750)
    dpi: int = 80
    
    # Plot settings
    num_lines_range: Tuple[int, int] = (1, 6)
    num_points_range: Tuple[int, int] = (5, 25)
    
    # Error bar settings (matching existing dataset)
    error_bar_probability: float = 0.7  # 70% of plots have error bars
    error_bar_min_pixels: float = 8.0   # Min error bar size in pixels (like existing: ~9px)
    error_bar_max_pixels: float = 150.0 # Max error bar size in pixels (like existing: ~150px)
    # Memory: run gc every N images to avoid laptop hang
    gc_interval: int = 50

config = GenerationConfig(
    output_dir="synthetic_dataset",
    num_images=3000,
    seed=42
)

print(f"Configuration:")
print(f"  Output: {config.output_dir}")
print(f"  Images: {config.num_images}")
print(f"  Error bar probability: {config.error_bar_probability}")
print(f"  Error bar range: {config.error_bar_min_pixels}-{config.error_bar_max_pixels} pixels")

Configuration:
  Output: synthetic_dataset
  Images: 3000
  Error bar probability: 0.7
  Error bar range: 8.0-150.0 pixels


## 2. Function Generators (PK-Specific)

In [33]:
class FunctionGenerator:
    """Generates PK-specific mathematical functions; uses learned param ranges from dataset if provided."""
    
    def __init__(self, param_ranges: Dict = None):
        self.param_ranges = param_ranges or {}
    
    def pk_single_dose_decay(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        x_range = max(float(x_norm.max() - x_norm.min()), 1e-6)
        if self.param_ranges.get("a") and self.param_ranges.get("k") and self.param_ranges.get("b"):
            k_learned = random.uniform(*self.param_ranges["k"])
            k = k_learned / x_range
            a_rand = random.uniform(*self.param_ranges["a"])
            b_rand = random.uniform(*self.param_ranges["b"])
            scale = random.uniform(100, 10000)
            C0 = scale * a_rand
            baseline = scale * max(b_rand, 0.05)
        else:
            C0 = random.uniform(100, 10000)
            k = random.uniform(0.01, 0.15)
            baseline = random.uniform(0.1, 10)
        y = C0 * np.exp(-k * x_norm) + baseline
        noise = np.random.normal(0, 0.05 * y)
        return np.maximum(y + noise, baseline)
    
    def pk_repeated_dosing(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        x_range = max(float(x_norm.max() - x_norm.min()), 1e-6)
        if self.param_ranges.get("k"):
            k = random.uniform(*self.param_ranges["k"]) / x_range
        else:
            k = random.uniform(0.02, 0.1)
        C0 = random.uniform(100, 5000)
        num_doses = random.randint(3, 8)
        dose_interval = x_range / num_doses
        y = np.zeros_like(x, dtype=float)
        for dose_num in range(num_doses):
            dose_time = dose_num * dose_interval
            mask = x_norm >= dose_time
            time_since_dose = x_norm[mask] - dose_time
            y[mask] += C0 * np.exp(-k * time_since_dose)
        baseline = random.uniform(1, 20)
        y += baseline
        noise = np.random.normal(0, 0.08 * y)
        return np.maximum(y + noise, baseline)
    
    def pk_two_compartment(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        A = random.uniform(500, 5000)
        B = random.uniform(100, 1000)
        alpha = random.uniform(0.1, 0.5)
        beta = random.uniform(0.01, 0.08)
        baseline = random.uniform(0.5, 5)
        y = A * np.exp(-alpha * x_norm) + B * np.exp(-beta * x_norm) + baseline
        noise = np.random.normal(0, 0.06 * y)
        return np.maximum(y + noise, baseline)
    
    def pk_absorption_elimination(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        ka = random.uniform(0.3, 1.5)
        ke = random.uniform(0.02, 0.1)
        Cmax = random.uniform(100, 2000)
        baseline = random.uniform(0.5, 10)
        if abs(ka - ke) < 0.01:
            ka = ke + 0.1
        y = Cmax * (ka / (ka - ke)) * (np.exp(-ke * x_norm) - np.exp(-ka * x_norm))
        y = np.maximum(y, 0) + baseline
        noise = np.random.normal(0, 0.07 * y)
        return np.maximum(y + noise, baseline)
    
    def pk_steady_state(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        x_range = x_norm.max()
        plateau = random.uniform(50, 500)
        amplitude = random.uniform(0.1, 0.3) * plateau
        period = x_range / random.randint(3, 7)
        rise_rate = random.uniform(0.05, 0.2)
        approach = 1 - np.exp(-rise_rate * x_norm)
        oscillation = amplitude * np.sin(2 * np.pi * x_norm / period)
        y = plateau * approach + oscillation
        noise = np.random.normal(0, 0.05 * plateau, len(x))
        return np.maximum(y + noise, plateau * 0.1)
    
    def pk_decay_to_plateau(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        x_range = max(float(x_norm.max() - x_norm.min()), 1e-6)
        if self.param_ranges.get("k"):
            k = random.uniform(*self.param_ranges["k"]) / x_range
        else:
            k = random.uniform(0.02, 0.1)
        C0 = random.uniform(500, 5000)
        Css = random.uniform(50, 300)
        y = (C0 - Css) * np.exp(-k * x_norm) + Css
        noise = np.random.normal(0, 0.05 * y)
        return np.maximum(y + noise, Css * 0.5)
    
    def linear_decline(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        C0 = random.uniform(100, 1000)
        rate = random.uniform(1, 20)
        baseline = random.uniform(5, 50)
        y = C0 - rate * x_norm
        y = np.maximum(y, baseline)
        noise = np.random.normal(0, 0.08 * C0, len(x))
        return np.maximum(y + noise, baseline)
    
    def flat_with_variation(self, x: np.ndarray) -> np.ndarray:
        baseline = random.uniform(20, 200)
        slight_trend = random.uniform(-0.5, 0.5)
        x_norm = x - x.min()
        y = baseline + slight_trend * x_norm
        noise = np.random.normal(0, 0.15 * baseline, len(x))
        return np.maximum(y + noise, baseline * 0.3)
    
    def biphasic_response(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        x_range = x_norm.max()
        baseline = random.uniform(10, 100)
        peak_time = random.uniform(0.2, 0.5) * x_range
        peak_height = random.uniform(1.5, 4) * baseline
        rise_rate = random.uniform(0.05, 0.2)
        fall_rate = random.uniform(0.02, 0.1)
        y = np.zeros_like(x, dtype=float)
        rise_mask = x_norm <= peak_time
        fall_mask = x_norm > peak_time
        y[rise_mask] = baseline + (peak_height - baseline) * (1 - np.exp(-rise_rate * x_norm[rise_mask]))
        time_after_peak = x_norm[fall_mask] - peak_time
        y[fall_mask] = baseline + (peak_height - baseline) * np.exp(-fall_rate * time_after_peak)
        noise = np.random.normal(0, 0.08 * baseline, len(x))
        return np.maximum(y + noise, baseline * 0.2)
    
    def log_linear_decay(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        C0 = random.uniform(1000, 10000)
        half_life = random.uniform(10, 100)
        k = np.log(2) / half_life
        baseline = random.uniform(1, 20)
        y = C0 * np.exp(-k * x_norm) + baseline
        noise = np.random.normal(0, 0.06 * y)
        return np.maximum(y + noise, baseline)
    
    def multiple_peak_decay(self, x: np.ndarray) -> np.ndarray:
        x_norm = x - x.min()
        x_range = x_norm.max()
        C0 = random.uniform(500, 3000)
        overall_decay = random.uniform(0.01, 0.05)
        num_peaks = random.randint(4, 8)
        y = np.zeros_like(x, dtype=float)
        for i in range(num_peaks):
            peak_time = (i + 0.5) * x_range / num_peaks
            peak_height = C0 * np.exp(-overall_decay * peak_time)
            local_decay = random.uniform(0.05, 0.15)
            for j, t in enumerate(x_norm):
                if t >= peak_time:
                    y[j] += peak_height * np.exp(-local_decay * (t - peak_time))
        baseline = random.uniform(5, 50)
        y += baseline
        noise = np.random.normal(0, 0.07 * y)
        return np.maximum(y + noise, baseline)
    
    def get_random_function(self) -> Callable:
        functions = [
            (self.pk_single_dose_decay, 3),
            (self.pk_repeated_dosing, 3),
            (self.pk_two_compartment, 2),
            (self.pk_absorption_elimination, 2),
            (self.pk_steady_state, 2),
            (self.pk_decay_to_plateau, 2),
            (self.linear_decline, 1),
            (self.flat_with_variation, 2),
            (self.biphasic_response, 2),
            (self.log_linear_decay, 2),
            (self.multiple_peak_decay, 2),
        ]
        total_weight = sum(w for _, w in functions)
        r = random.uniform(0, total_weight)
        cumulative = 0
        for func, weight in functions:
            cumulative += weight
            if r <= cumulative:
                return func
        return functions[0][0]

print("FunctionGenerator defined with 11 PK-specific functions")

FunctionGenerator defined with 11 PK-specific functions


## 3. Plot Style Generator

In [34]:
class PlotStyleGenerator:
    LINE_STYLES = ['-', '--', '-.', ':']
    MARKERS = ['o', 's', '^', 'v', '<', '>', 'D', 'p', 'h', '*', 'X', 'P', 'd', '8']
    COLORS = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
        '#000000', '#0000FF', '#FF0000', '#00FF00', '#800080'
    ]
    
    LINE_NAME_PREFIXES = ["Treatment", "Control", "Placebo", "Drug", "Cohort", 
                         "Infusion", "Dose", "Patient", "HbA1c", "Subject"]
    LINE_NAME_SUFFIXES = ["A", "B", "C", "1", "2", "3", "ER", "10mg", "50mg", "100mg",
                         "Day 1", "Day 7", "Week 4", "Week 12"]
    
    X_LABELS = ["Time (hours)", "Time (days)", "Days post infusion", "Week", "Day"]
    Y_LABELS = ["Concentration (ng/mL)", "Plasma concentration (ng/mL)", 
                "Log Concentration ng/ml", "Response (%)", "HbA1c (%)"]
    
    @staticmethod
    def generate_line_name() -> str:
        prefix = random.choice(PlotStyleGenerator.LINE_NAME_PREFIXES)
        suffix = random.choice(PlotStyleGenerator.LINE_NAME_SUFFIXES)
        fmt = random.choice([f"{prefix}_{suffix}", f"{prefix} {suffix}", f"{prefix}-{suffix}"])
        return fmt
    
    @staticmethod
    def get_distinct_styles(num_lines: int) -> List[Dict]:
        styles = []
        colors = PlotStyleGenerator.COLORS.copy()
        markers = PlotStyleGenerator.MARKERS.copy()
        random.shuffle(colors)
        random.shuffle(markers)
        for i in range(num_lines):
            color = colors[i % len(colors)]
            styles.append({
                'color': color,
                'marker': markers[i % len(markers)],
                'linestyle': PlotStyleGenerator.LINE_STYLES[i % 4],
                'linewidth': random.uniform(1.0, 2.5),
                'markersize': random.uniform(5, 10),
                'markerfacecolor': color if random.random() > 0.3 else 'white',
                'markeredgecolor': color,
                'markeredgewidth': random.uniform(1, 2)
            })
        return styles

print("PlotStyleGenerator defined")

PlotStyleGenerator defined


In [ ]:
try:
    from scipy.optimize import curve_fit
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False
    print("Install scipy for curve fitting: pip install scipy")

def exp_decay(x, a, k, b):
    return a * np.exp(-k * x) + b

def learn_from_dataset(labels_dir: str = "dataset/labels") -> Dict:
    """Load existing labels, convert pixel->data, fit exponential; return param ranges."""
    labels_path = Path(labels_dir)
    if not labels_path.exists():
        print(f"Not found: {labels_dir}. Skip learning.")
        return {}
    all_a, all_k, all_b = [], [], []
    file_count, line_count = 0, 0
    for jpath in sorted(labels_path.glob("*.json")):
        with open(jpath) as f:
            data = json.load(f)
        for line_obj in data:
            pts = line_obj["points"]
            data_pts = [p for p in pts if p["label"] == ""]
            axis_pts = {p["label"]: (p["x"], p["y"]) for p in pts if p["label"] in ("xmin","xmax","ymin","ymax")}
            if len(axis_pts) < 4 or len(data_pts) < 4:
                continue
            xmin_px, ymin_px = axis_pts["xmin"][0], axis_pts["ymin"][1]
            xmax_px, ymax_px = axis_pts["xmax"][0], axis_pts["ymax"][1]
            dx = xmax_px - xmin_px
            dy = ymin_px - ymax_px  # y increases upward in data
            if dx < 1 or abs(dy) < 1:
                continue
            x_norm = np.array([(p["x"] - xmin_px) / dx for p in data_pts])
            y_norm = np.array([(p["y"] - ymax_px) / dy for p in data_pts])
            x_norm = np.clip(x_norm, 0, 1)
            y_norm = np.clip(y_norm, 0, 1)
            if not HAS_SCIPY:
                continue
            try:
                (a, k, b), _ = curve_fit(exp_decay, x_norm, y_norm, p0=[1, 2, 0], maxfev=2000,
                                         bounds=([0.01, 0.01, -0.5], [10, 20, 1.5]))
                all_a.append(a); all_k.append(k); all_b.append(b)
                line_count += 1
            except Exception:
                pass
        file_count += 1
    if not all_a:
        print("No fits succeeded. Use default generator params.")
        return {}
    ranges = {
        "a": (float(np.min(all_a)), float(np.max(all_a))),
        "k": (float(np.min(all_k)), float(np.max(all_k))),
        "b": (float(np.min(all_b)), float(np.max(all_b))),
        "n_files": file_count,
        "n_lines_fit": line_count,
    }
    with open("dataset_param_ranges.json", "w") as f:
        json.dump(ranges, f, indent=2)
    print(f"Learned from {file_count} files, {line_count} lines. Param ranges saved to dataset_param_ranges.json")
    print("a:", ranges["a"], "k:", ranges["k"], "b:", ranges["b"])
    return ranges

def learn_error_bar_ranges_from_dataset(labels_dir: str = "dataset/labels") -> Dict:
    labels_path = Path(labels_dir)
    if not labels_path.exists():
        return {}
    all_top, all_bottom, all_dev = [], [], []
    for jpath in sorted(labels_path.glob("*.json")):
        with open(jpath) as f:
            data = json.load(f)
        for line_obj in data:
            for p in line_obj.get("points", []):
                if p.get("label") != "":
                    continue
                all_top.append(float(p.get("topBarPixelDistance", 0)))
                all_bottom.append(float(p.get("bottomBarPixelDistance", 0)))
                all_dev.append(float(p.get("deviationPixelDistance", 0)))
    if not all_top:
        return {}
    ranges = {
        "topBar_min": float(np.min(all_top)), "topBar_max": float(np.max(all_top)),
        "bottomBar_min": float(np.min(all_bottom)), "bottomBar_max": float(np.max(all_bottom)),
        "deviation_min": float(np.min(all_dev)), "deviation_max": float(np.max(all_dev)),
        "n_data_points": len(all_top),
    }
    with open("dataset_error_bar_ranges.json", "w") as f:
        json.dump(ranges, f, indent=2)
    print("Error bar ranges from 150 labels saved to dataset_error_bar_ranges.json")
    print("  topBar:", ranges["topBar_min"], "-", ranges["topBar_max"], "| bottomBar:", ranges["bottomBar_min"], "-", ranges["bottomBar_max"])
    print("  deviationPixelDistance:", ranges["deviation_min"], "-", ranges["deviation_max"])
    return ranges

DATASET_PARAM_RANGES = learn_from_dataset() if Path("dataset/labels").exists() else {}
DATASET_ERROR_BAR_RANGES = learn_error_bar_ranges_from_dataset() if Path("dataset/labels").exists() else {}

Learned from 150 files, 484 lines. Param ranges saved to dataset_param_ranges.json
a: (0.010000000000000002, 9.9999997405937) k: (0.010000000000000002, 19.999999999999996) b: (-0.49999999999999994, 0.9900427397058262)
Error bar ranges from 150 labels saved to dataset_error_bar_ranges.json
  topBar: 0.0 - 545.0 | bottomBar: 0.0 - 537.0
  deviationPixelDistance: 0.0 - 545.0


## 4. Annotation Generator

In [36]:
class AnnotationGenerator:
    @staticmethod
    def create_point_annotation(x: float, y: float, label: str = "",
                                top_bar: float = 0, bottom_bar: float = 0) -> Dict:
        deviation = max(top_bar, bottom_bar)
        return {
            "x": float(x),
            "y": float(y),
            "label": label,
            "topBarPixelDistance": float(top_bar),
            "bottomBarPixelDistance": float(bottom_bar),
            "deviationPixelDistance": float(deviation)
        }
    
    @staticmethod
    def create_line_annotation(line_name: str, points: List[Dict]) -> Dict:
        return {
            "label": {"lineName": line_name},
            "points": points
        }

print("AnnotationGenerator defined")
print("Sample:", json.dumps(AnnotationGenerator.create_point_annotation(100.5, 200.3, "", 9.5, 12.3), indent=2))

AnnotationGenerator defined
Sample: {
  "x": 100.5,
  "y": 200.3,
  "label": "",
  "topBarPixelDistance": 9.5,
  "bottomBarPixelDistance": 12.3,
  "deviationPixelDistance": 12.3
}


## 5. Main Plot Generator

In [ ]:
class SyntheticPlotGenerator:
    def __init__(self, config: GenerationConfig, param_ranges: Dict = None):
        self.config = config
        if param_ranges is None:
            param_ranges_path = Path("dataset_param_ranges.json")
            if param_ranges_path.exists():
                with open(param_ranges_path) as f:
                    pr = json.load(f)
                param_ranges = {k: tuple(v) for k, v in pr.items() if k in ("a", "k", "b") and isinstance(v, (list, tuple)) and len(v) == 2}
            else:
                param_ranges = {}
        self.function_gen = FunctionGenerator(param_ranges=param_ranges)
        self.style_gen = PlotStyleGenerator()
        self.annotation_gen = AnnotationGenerator()
        
        self.images_dir = Path(config.output_dir) / "images"
        self.labels_dir = Path(config.output_dir) / "labels"
        self.images_dir.mkdir(parents=True, exist_ok=True)
        self.labels_dir.mkdir(parents=True, exist_ok=True)
        
        random.seed(config.seed)
        np.random.seed(config.seed)
    
    def generate_single_plot(self) -> Tuple[str, List]:
        plot_id = str(uuid.uuid4())
        
        width = random.randint(*self.config.image_width_range)
        height = random.randint(*self.config.image_height_range)
        figsize = (width / self.config.dpi, height / self.config.dpi)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=self.config.dpi)
        
        num_lines = random.randint(*self.config.num_lines_range)
        x_start = 0
        x_end = random.choice([28, 56, 84, 112, 168, 252, 365])
        
        styles = self.style_gen.get_distinct_styles(num_lines)
        all_pixel_points = []
        
        plot_has_error_bars = random.random() < self.config.error_bar_probability
        
        for line_idx in range(num_lines):
            num_points = random.randint(*self.config.num_points_range)
            x_data = np.linspace(x_start, x_end, num_points)
            
            func = self.function_gen.get_random_function()
            y_data = func(x_data)
            
            # Error bars:config.error_bar_min/max_pixels
            if plot_has_error_bars:
                y_range = max(y_data.max() - y_data.min(), 1)
                # Target pixel range from 150 labels -> fraction of y_range (height used for scale)
                eb_min, eb_max = self.config.error_bar_min_pixels, self.config.error_bar_max_pixels
                frac_lo = max(0.01, eb_min * 1.15 / height)
                frac_hi = min(0.5, eb_max * 1.15 / height)
                if frac_hi <= frac_lo:
                    frac_lo, frac_hi = 0.05, 0.4
                error_top = np.abs(np.random.uniform(frac_lo, frac_hi, num_points)) * y_range
                error_bottom = np.abs(np.random.uniform(frac_lo, frac_hi, num_points)) * y_range
            else:
                error_top = np.zeros(num_points)
                error_bottom = np.zeros(num_points)
            
            style = styles[line_idx]
            line_name = self.style_gen.generate_line_name()
            
            if plot_has_error_bars:
                ax.errorbar(x_data, y_data, yerr=[error_bottom, error_top],
                           color=style['color'], marker=style['marker'],
                           linestyle=style['linestyle'], linewidth=style['linewidth'],
                           markersize=style['markersize'],
                           markerfacecolor=style['markerfacecolor'],
                           markeredgecolor=style['markeredgecolor'],
                           markeredgewidth=style['markeredgewidth'],
                           capsize=3, label=line_name)
            else:
                ax.plot(x_data, y_data, color=style['color'], marker=style['marker'],
                       linestyle=style['linestyle'], linewidth=style['linewidth'],
                       markersize=style['markersize'],
                       markerfacecolor=style['markerfacecolor'],
                       markeredgecolor=style['markeredgecolor'],
                       markeredgewidth=style['markeredgewidth'],
                       label=line_name)
            
            all_pixel_points.append({
                'line_name': line_name,
                'x_data': x_data,
                'y_data': y_data,
                'error_top': error_top,
                'error_bottom': error_bottom,
                'has_error_bars': plot_has_error_bars
            })
        
        ax.set_xlabel(random.choice(self.style_gen.X_LABELS), fontsize=random.randint(10, 14))
        ax.set_ylabel(random.choice(self.style_gen.Y_LABELS), fontsize=random.randint(10, 14))
        
        if random.random() > 0.3:
            ax.legend(loc=random.choice(['best', 'upper right', 'upper left']), fontsize=random.randint(8, 12))
        
        if random.random() > 0.5:
            ax.grid(True, alpha=random.uniform(0.2, 0.5))
        
        plt.tight_layout()
        
        image_path = self.images_dir / f"{plot_id}.png"
        fig.savefig(image_path, dpi=self.config.dpi, bbox_inches='tight',
                   facecolor='white', edgecolor='none')
        
        with Image.open(image_path) as img:
            actual_width, actual_height = img.size
        
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        tight_bbox = fig.get_tightbbox(renderer)
        
        scale_x = actual_width / (tight_bbox.width * self.config.dpi)
        scale_y = actual_height / (tight_bbox.height * self.config.dpi)
        
        all_annotations = []
        for line_data in all_pixel_points:
            points = []
            
            for i in range(len(line_data['x_data'])):
                display_coords = ax.transData.transform(
                    (line_data['x_data'][i], line_data['y_data'][i])
                )
                
                x_pixel = (display_coords[0] - tight_bbox.x0 * self.config.dpi) * scale_x
                y_pixel = actual_height - (display_coords[1] - tight_bbox.y0 * self.config.dpi) * scale_y
                
                
                if line_data['has_error_bars'] and (line_data['error_top'][i] > 0 or line_data['error_bottom'][i] > 0):
                    top_coords = ax.transData.transform(
                        (line_data['x_data'][i], line_data['y_data'][i] + line_data['error_top'][i])
                    )
                    top_pixel_dist = abs(display_coords[1] - top_coords[1]) * scale_y
                    
                    bottom_coords = ax.transData.transform(
                        (line_data['x_data'][i], line_data['y_data'][i] - line_data['error_bottom'][i])
                    )
                    bottom_pixel_dist = abs(display_coords[1] - bottom_coords[1]) * scale_y
                else:
                    top_pixel_dist = 0
                    bottom_pixel_dist = 0
                
                points.append(self.annotation_gen.create_point_annotation(
                    x_pixel, y_pixel, "", top_pixel_dist, bottom_pixel_dist
                ))
            
            # Add axis boundary points (xmin, xmax, ymin, ymax)
            x_lim = ax.get_xlim()
            y_lim = ax.get_ylim()
            
            ymin_display = ax.transData.transform((x_lim[0], y_lim[0]))
            ymin_x = (ymin_display[0] - tight_bbox.x0 * self.config.dpi) * scale_x
            ymin_y = actual_height - (ymin_display[1] - tight_bbox.y0 * self.config.dpi) * scale_y
            
            ymax_display = ax.transData.transform((x_lim[0], y_lim[1]))
            ymax_x = (ymax_display[0] - tight_bbox.x0 * self.config.dpi) * scale_x
            ymax_y = actual_height - (ymax_display[1] - tight_bbox.y0 * self.config.dpi) * scale_y
            
            xmax_display = ax.transData.transform((x_lim[1], y_lim[0]))
            xmax_x = (xmax_display[0] - tight_bbox.x0 * self.config.dpi) * scale_x
            xmax_y = actual_height - (xmax_display[1] - tight_bbox.y0 * self.config.dpi) * scale_y
            
            points.append(self.annotation_gen.create_point_annotation(ymin_x, ymin_y, "ymin"))
            points.append(self.annotation_gen.create_point_annotation(ymax_x, ymax_y, "ymax"))
            points.append(self.annotation_gen.create_point_annotation(ymin_x, ymin_y, "xmin"))
            points.append(self.annotation_gen.create_point_annotation(xmax_x, xmax_y, "xmax"))
            
            all_annotations.append(self.annotation_gen.create_line_annotation(
                line_data['line_name'], points
            ))
        
        plt.close(fig)
        
        label_path = self.labels_dir / f"{plot_id}.json"
        with open(label_path, 'w') as f:
            json.dump(all_annotations, f, indent=2)
        
        return plot_id, all_annotations
    
    def generate_dataset(self) -> List[str]:
        generated_ids = []
        print(f"Generating {self.config.num_images} images...")
        print(f"Output: {self.config.output_dir}")
        
        gc_interval = getattr(self.config, 'gc_interval', 50)
        for i in tqdm(range(self.config.num_images), desc="Generating"):
            try:
                plot_id, _ = self.generate_single_plot()
                generated_ids.append(plot_id)
            except Exception as e:
                print(f"Error at {i}: {e}")
                continue
            if (i + 1) % gc_interval == 0:
                gc.collect()
        gc.collect()
        print(f"\nComplete! Generated {len(generated_ids)} images.")
        return generated_ids

print("SyntheticPlotGenerator defined!")

SyntheticPlotGenerator defined!


## 6. Test Single Generation

In [38]:
# Test with single image
test_config = GenerationConfig(output_dir="test_output", num_images=1, seed=42)
test_gen = SyntheticPlotGenerator(test_config)
plot_id, annotations = test_gen.generate_single_plot()

print(f"Generated: {plot_id}")
print(f"Lines: {len(annotations)}")
print("\nFirst few points:")
for point in annotations[0]['points'][:3]:
    print(f"  x={point['x']:.1f}, y={point['y']:.1f}, top={point['topBarPixelDistance']:.1f}, bottom={point['bottomBarPixelDistance']:.1f}")

Generated: df38ad28-e675-4504-a2e6-d8cbb414f412
Lines: 1

First few points:
  x=88.9, y=370.0, top=6.5, bottom=41.4
  x=201.7, y=107.9, top=86.9, bottom=29.4
  x=314.5, y=306.5, top=75.3, bottom=56.6


In [39]:
# Display test image
img = Image.open(f"test_output/images/{plot_id}.png")
plt.figure(figsize=(12, 8))
plt.imshow(img)
plt.title(f"Test: {plot_id[:8]}...")
plt.axis('off')
plt.show()
print(f"Size: {img.size}")

Size: (918, 498)


## 7. Generate Full Dataset (~3000 images)

In [ ]:
def _load_error_bar_ranges() -> Tuple[float, float]:
    p = Path("dataset_error_bar_ranges.json")
    if not p.exists():
        return 8.0, 150.0
    with open(p) as f:
        r = json.load(f)
    return float(r.get("deviation_min", 8)), float(r.get("deviation_max", 150))

eb_min, eb_max = _load_error_bar_ranges()
full_config = GenerationConfig(
    output_dir="synthetic_dataset",
    num_images=3000,
    seed=42,
    error_bar_probability=0.7,  # 70% like 150 dataset
    error_bar_min_pixels=eb_min,
    error_bar_max_pixels=eb_max,
)

print(f"Will generate {full_config.num_images} images")
print(f"Output: {full_config.output_dir}/")
print(f"Error bar range (from 150 labels): {eb_min}-{eb_max} px")

Will generate 3000 images
Output: synthetic_dataset/
Error bar range (from 150 labels): 0.0-545.0 px


In [41]:
# running 3000 at once may hang. Use the batch cell below (500 x 6) instead.
# generator = SyntheticPlotGenerator(full_config)
# generated_ids = generator.generate_dataset()

In [50]:

BATCH_SIZE = 500
start_idx = len(list(Path(full_config.output_dir).glob("images/*.png")))
if start_idx >= full_config.num_images:
    print("Already generated", start_idx, "images. Done.")
else:
    n_this_time = min(BATCH_SIZE, full_config.num_images - start_idx)
    print(f"Generating batch: {n_this_time} images (total so far: {start_idx})")
    gen_batch = SyntheticPlotGenerator(full_config)
    for _ in tqdm(range(n_this_time), desc="Batch"):
        try:
            gen_batch.generate_single_plot()
        except Exception as e:
            print("Error:", e)
        if (_ + 1) % 50 == 0:
            gc.collect()
    gc.collect()
    now = len(list(Path(full_config.output_dir).glob("images/*.png")))
    print(f"Done. Total images now: {now}")

Generating batch: 500 images (total so far: 2500)


Batch: 100%|██████████| 500/500 [03:56<00:00,  2.11it/s]

Done. Total images now: 3000


In [51]:
# Check generated files
images_dir = Path(full_config.output_dir) / "images"
labels_dir = Path(full_config.output_dir) / "labels"

num_images = len(list(images_dir.glob("*.png")))
num_labels = len(list(labels_dir.glob("*.json")))

print(f"Images: {num_images}")
print(f"Labels: {num_labels}")
print(f"Match: {'Yes' if num_images == num_labels else 'No'}")

# Check a random label for error bar values
sample_label = list(labels_dir.glob("*.json"))[0]
with open(sample_label) as f:
    sample = json.load(f)
print(f"\nSample label ({sample_label.name}):")
for point in sample[0]['points'][:3]:
    print(f"  top={point['topBarPixelDistance']:.1f}, bottom={point['bottomBarPixelDistance']:.1f}")

Images: 3000
Labels: 3000
Match: Yes

Sample label (0002e3ca-7bbc-4560-a06a-483521978679.json):
  top=55.7, bottom=114.1
  top=17.8, bottom=64.1
  top=97.6, bottom=106.4


In [52]:
# Display random samples
all_images = list(images_dir.glob("*.png"))
samples = random.sample(all_images, min(6, len(all_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flatten(), samples):
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_path.stem[:8] + "...", fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [53]:
from pathlib import Path

folder = Path("synthetic_dataset/labels")

file_count = sum(p.is_file() for p in folder.iterdir())

print(file_count)


3000
